### MINI-PROJET 3IDL : Détection d'Émotions dans les Textes avec Deep Learning

### Réalisé par :
  - **Ahmed Takieddine Ghrib**
  - **Ghassen Mastouri**
  - **Nessim Zemzem**

**Classe: 3 IDL 02**


### SECTION 1 : INSTALLATION ET IMPORTS

In [1]:
# Installation des bibliothèques nécessaires
!pip install -q transformers datasets lime shap plotly scikit-learn-extra
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 10.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.0/819.0 kB 46.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# Imports
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_fscore_support,
    hamming_loss,
    roc_auc_score,
    multilabel_confusion_matrix,
    classification_report
)
from sklearn.preprocessing import MultiLabelBinarizer

from transformers import (
    BertTokenizer,
    BertModel,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW

import nltk
nltk.download('punkt')
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Vérification GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Device utilisé : {device}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"Mémoire GPU : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


🚀 Device utilisé : cuda
GPU : Tesla T4
Mémoire GPU : 15.83 GB


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


### SECTION 2 : CHARGEMENT ET EXPLORATION DU DATASET

In [3]:
print("\n📥 TÉLÉCHARGEMENT DU DATASET GOEMOTIONS...")

urls = {
    'train': 'https://raw.githubusercontent.com/google-research/google-research/master/goemotions/data/train.tsv',
    'dev': 'https://raw.githubusercontent.com/google-research/google-research/master/goemotions/data/dev.tsv',
    'test': 'https://raw.githubusercontent.com/google-research/google-research/master/goemotions/data/test.tsv'
}

for split, url in urls.items():
    filename = f'{split}.tsv'
    if not os.path.exists(filename):
        print(f"Téléchargement de {split}.tsv...")
        urllib.request.urlretrieve(url, filename)
    else:
        print(f"{split}.tsv déjà téléchargé")

# Chargement des données
df_train = pd.read_csv('train.tsv', sep='\t', header=None,
                        names=['text', 'labels', 'id'])
df_dev = pd.read_csv('dev.tsv', sep='\t', header=None,
                      names=['text', 'labels', 'id'])
df_test = pd.read_csv('test.tsv', sep='\t', header=None,
                       names=['text', 'labels', 'id'])

print(f"\n📊 STATISTIQUES DU DATASET :")
print(f"Train : {len(df_train)} échantillons")
print(f"Dev : {len(df_dev)} échantillons")
print(f"Test : {len(df_test)} échantillons")
print(f"Total : {len(df_train) + len(df_dev) + len(df_test)} échantillons")

# Les 28 émotions de GoEmotions (27 + neutral)
EMOTIONS = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
    'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
    'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
    'relief', 'remorse', 'sadness', 'surprise', 'neutral'
]

print(f"\n🎭 Nombre d'émotions : {len(EMOTIONS)}")
print(f"Émotions : {', '.join(EMOTIONS)}")



📥 TÉLÉCHARGEMENT DU DATASET GOEMOTIONS...
Téléchargement de train.tsv...
Téléchargement de dev.tsv...
Téléchargement de test.tsv...

📊 STATISTIQUES DU DATASET :
Train : 43410 échantillons
Dev : 5426 échantillons
Test : 5427 échantillons
Total : 54263 échantillons

🎭 Nombre d'émotions : 28
Émotions : admiration, amusement, anger, annoyance, approval, caring, confusion, curiosity, desire, disappointment, disapproval, disgust, embarrassment, excitement, fear, gratitude, grief, joy, love, nervousness, optimism, pride, realization, relief, remorse, sadness, surprise, neutral


### SECTION 3 : PRÉTRAITEMENT ET ANALYSE EXPLORATOIRE

In [4]:
print("\n🔍 ANALYSE EXPLORATOIRE...")

def parse_labels(label_str):
    """Convertit une chaîne de labels en liste d'entiers"""
    if pd.isna(label_str) or label_str == '':
        return []
    return [int(x) for x in str(label_str).split(',')]

# Parser les labels
for df in [df_train, df_dev, df_test]:
    df['label_list'] = df['labels'].apply(parse_labels)
    df['num_labels'] = df['label_list'].apply(len)

# Statistiques sur les labels
print(f"\n📈 Distribution du nombre de labels par texte :")
print(df_train['num_labels'].value_counts().sort_index())

# Visualisation de la distribution des émotions
label_counts = {}
for labels in df_train['label_list']:
    for label in labels:
        if label < len(EMOTIONS):
            label_counts[EMOTIONS[label]] = label_counts.get(EMOTIONS[label], 0) + 1

# Top 15 émotions
top_emotions = sorted(label_counts.items(), key=lambda x: x[1], reverse=True)[:15]

fig = go.Figure([go.Bar(
    x=[e[0] for e in top_emotions],
    y=[e[1] for e in top_emotions],
    marker_color='lightblue'
)])
fig.update_layout(
    title='Distribution des 15 émotions les plus fréquentes (Train Set)',
    xaxis_title='Émotion',
    yaxis_title='Fréquence',
    height=500
)
fig.show()

# Analyse de la longueur des textes
df_train['text_length'] = df_train['text'].apply(lambda x: len(str(x).split()))

fig = go.Figure([go.Histogram(
    x=df_train['text_length'],
    nbinsx=50,
    marker_color='coral'
)])
fig.update_layout(
    title='Distribution de la longueur des textes (en mots)',
    xaxis_title='Nombre de mots',
    yaxis_title='Fréquence',
    height=400
)
fig.show()

print(f"\n📝 Statistiques de longueur des textes :")
print(df_train['text_length'].describe())


🔍 ANALYSE EXPLORATOIRE...

📈 Distribution du nombre de labels par texte :
num_labels
1    36308
2     6541
3      532
4       28
5        1
Name: count, dtype: int64



📝 Statistiques de longueur des textes :
count    43410.000000
mean        12.840175
std          6.701597
min          1.000000
25%          7.000000
50%         12.000000
75%         18.000000
max         33.000000
Name: text_length, dtype: float64


### SECTION 4 : PRÉPARATION DES DONNÉES POUR L'ENTRAÎNEMENT

In [5]:
print("\n🛠️ PRÉPARATION DES DONNÉES...")

class GoEmotionsDataset(Dataset):
    """Dataset PyTorch pour GoEmotions"""

    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        labels = self.labels[idx]

        # Tokenization
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        # One-hot encoding des labels
        label_vector = torch.zeros(len(EMOTIONS))
        for label in labels:
            if label < len(EMOTIONS):
                label_vector[label] = 1.0

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': label_vector
        }

# Initialisation du tokenizer BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Préparation des datasets
MAX_LEN = 128
BATCH_SIZE = 32 if torch.cuda.is_available() else 8

train_dataset = GoEmotionsDataset(
    df_train['text'].values,
    df_train['label_list'].values,
    tokenizer,
    MAX_LEN
)

dev_dataset = GoEmotionsDataset(
    df_dev['text'].values,
    df_dev['label_list'].values,
    tokenizer,
    MAX_LEN
)

test_dataset = GoEmotionsDataset(
    df_test['text'].values,
    df_test['label_list'].values,
    tokenizer,
    MAX_LEN
)

# DataLoaders optimisés
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"✅ Datasets créés avec succès !")
print(f"Batch size : {BATCH_SIZE}")
print(f"Nombre de batches (train) : {len(train_loader)}")



🛠️ PRÉPARATION DES DONNÉES...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

✅ Datasets créés avec succès !
Batch size : 32
Nombre de batches (train) : 1357


### SECTION 5 : ARCHITECTURES DE MODÈLES

### MODÈLE 1 : LSTM Simple

In [6]:
class SimpleLSTM(nn.Module):
    """LSTM simple pour classification multi-label"""

    def __init__(self, vocab_size, embedding_dim=300, hidden_dim=256,
                 num_classes=28, dropout=0.3):
        super(SimpleLSTM, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            dropout=dropout
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids, attention_mask=None):
        embedded = self.embedding(input_ids)
        lstm_out, (hidden, cell) = self.lstm(embedded)

        # Utiliser le dernier état caché
        output = self.dropout(hidden[-1])
        logits = self.fc(output)

        return logits


### MODÈLE 2 : BiLSTM avec Attention

In [7]:
class AttentionLayer(nn.Module):
    """Mécanisme d'attention"""

    def __init__(self, hidden_dim):
        super(AttentionLayer, self).__init__()
        self.attention = nn.Linear(hidden_dim * 2, 1)

    def forward(self, lstm_output):
        # lstm_output: (batch, seq_len, hidden*2)
        attention_weights = torch.softmax(
            self.attention(lstm_output).squeeze(-1),
            dim=1
        )
        # attention_weights: (batch, seq_len)

        # Contexte pondéré
        context = torch.sum(
            attention_weights.unsqueeze(-1) * lstm_output,
            dim=1
        )

        return context, attention_weights

class BiLSTMAttention(nn.Module):
    """BiLSTM avec mécanisme d'attention"""

    def __init__(self, vocab_size, embedding_dim=300, hidden_dim=256,
                 num_classes=28, dropout=0.3):
        super(BiLSTMAttention, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.bilstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True,
            dropout=dropout,
            num_layers=2
        )
        self.attention = AttentionLayer(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids, attention_mask=None):
        embedded = self.embedding(input_ids)
        lstm_out, _ = self.bilstm(embedded)

        # Appliquer l'attention
        context, attn_weights = self.attention(lstm_out)
        output = self.dropout(context)
        logits = self.fc(output)

        return logits


### MODÈLE 3 : CNN-BiLSTM Hybride avec Attention

In [8]:
# 🔹 Couche Attention
class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, 1)

    def forward(self, lstm_out):
        # lstm_out: (batch, seq_len, hidden*2)
        attn_weights = torch.softmax(self.attn(lstm_out).squeeze(-1), dim=1)
        context = torch.sum(lstm_out * attn_weights.unsqueeze(-1), dim=1)
        return context, attn_weights

# 🔹 Modèle CNN-BiLSTM avec Attention (corrigé)
class CNNBiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, embedding_dim=300, hidden_dim=256,
                 num_classes=28, dropout=0.3, num_filters=100):
        super().__init__()

        # Embedding
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        # CNN
        self.conv1 = nn.Conv1d(embedding_dim, num_filters, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(embedding_dim, num_filters, kernel_size=4, padding=2)
        self.conv3 = nn.Conv1d(embedding_dim, num_filters, kernel_size=5, padding=2)

        # BiLSTM
        self.bilstm = nn.LSTM(
            input_size=num_filters * 3,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True,
            num_layers=2,
            dropout=dropout
        )

        # Attention
        self.attention = AttentionLayer(hidden_dim)

        # Classification
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids, attention_mask=None):
        # Embedding
        x = self.embedding(input_ids)           # (batch, seq_len, embed_dim)
        x = x.transpose(1, 2)                   # (batch, embed_dim, seq_len)

        # CNN
        conv1_out = F.relu(self.conv1(x))
        conv2_out = F.relu(self.conv2(x))
        conv3_out = F.relu(self.conv3(x))

        # 🔹 Troncature pour assurer la même longueur seq_len
        min_len = min(conv1_out.size(2), conv2_out.size(2), conv3_out.size(2))
        conv1_out = conv1_out[:, :, :min_len]
        conv2_out = conv2_out[:, :, :min_len]
        conv3_out = conv3_out[:, :, :min_len]

        # Concaténation
        conv_out = torch.cat([conv1_out, conv2_out, conv3_out], dim=1)
        conv_out = conv_out.transpose(1, 2)    # (batch, seq_len, num_filters*3)

        # BiLSTM
        lstm_out, _ = self.bilstm(conv_out)

        # Attention
        context, attn_weights = self.attention(lstm_out)

        # Classification
        context = self.dropout(context)
        logits = self.fc(context)

        return logits

### MODÈLE 4 : BERT Fine-tuned

In [9]:
class BERTEmotionClassifier(nn.Module):
    """BERT fine-tuné pour classification multi-label"""

    def __init__(self, num_classes=28, dropout=0.3):
        super(BERTEmotionClassifier, self).__init__()

        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(768, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Utiliser le [CLS] token
        pooled_output = outputs.pooler_output
        output = self.dropout(pooled_output)
        logits = self.classifier(output)

        return logits

print("✅ Architectures définies !")


✅ Architectures définies !


### SECTION 6 : FONCTIONS D'ENTRAÎNEMENT ET D'ÉVALUATION

In [10]:
print("\n⚙️ DÉFINITION DES FONCTIONS D'ENTRAÎNEMENT...")

def train_epoch(model, data_loader, criterion, optimizer, device, scaler=None):
    """Entraîne le modèle pour une époque"""
    model.train()
    total_loss = 0

    for batch in data_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        # Mixed precision training
        if scaler:
            with autocast():
                outputs = model(input_ids, attention_mask)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        total_loss += loss.item()

    return total_loss / len(data_loader)

def evaluate_model(model, data_loader, criterion, device, threshold=0.5):
    """Évalue le modèle"""
    model.eval()
    total_loss = 0
    all_predictions = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Prédictions
            probs = torch.sigmoid(outputs)
            preds = (probs > threshold).float()

            all_predictions.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    # Concaténer tous les résultats
    all_predictions = np.vstack(all_predictions)
    all_labels = np.vstack(all_labels)
    all_probs = np.vstack(all_probs)

    # Calculer les métriques
    precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(
        all_labels, all_predictions, average='micro', zero_division=0
    )
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        all_labels, all_predictions, average='macro', zero_division=0
    )

    hamming = hamming_loss(all_labels, all_predictions)

    # Calculer l'accuracy
    # Pour multi-label, on utilise subset accuracy (toutes les labels doivent correspondre exactement)
    subset_accuracy = np.mean([np.array_equal(pred, true)
                               for pred, true in zip(all_predictions, all_labels)])

    # Accuracy par label (combien de labels individuels sont correctement prédits)
    label_accuracy = np.mean(all_predictions == all_labels)

    # AUC-ROC (gérer les cas où une classe n'a aucun exemple)
    try:
        auc_roc = roc_auc_score(all_labels, all_probs, average='macro')
    except:
        auc_roc = 0.0

    metrics = {
        'loss': total_loss / len(data_loader),
        'precision_micro': precision_micro,
        'recall_micro': recall_micro,
        'f1_micro': f1_micro,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        'hamming_loss': hamming,
        'subset_accuracy': subset_accuracy,
        'label_accuracy': label_accuracy,
        'auc_roc': auc_roc
    }

    return metrics

def train_model(model, train_loader, dev_loader, num_epochs=10,
                lr=2e-5, model_name='model', patience=3):
    """Entraîne un modèle complet avec early stopping"""

    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    # Learning rate scheduler
    total_steps = len(train_loader) * num_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=total_steps // 10,
        num_training_steps=total_steps
    )

    # Mixed precision scaler
    scaler = GradScaler() if torch.cuda.is_available() else None

    # Early stopping
    best_f1 = 0
    patience_counter = 0

    history = {
        'train_loss': [],
        'val_loss': [],
        'val_f1_micro': [],
        'val_f1_macro': [],
        'val_subset_accuracy': [],
        'val_label_accuracy': []
    }

    print(f"\n🚀 Début de l'entraînement du modèle {model_name}...")

    for epoch in range(num_epochs):
        # Entraînement
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device, scaler)
        scheduler.step()

        # Évaluation
        val_metrics = evaluate_model(model, dev_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_metrics['loss'])
        history['val_f1_micro'].append(val_metrics['f1_micro'])
        history['val_f1_macro'].append(val_metrics['f1_macro'])
        history['val_subset_accuracy'].append(val_metrics['subset_accuracy'])
        history['val_label_accuracy'].append(val_metrics['label_accuracy'])

        print(f"Epoch {epoch+1}/{num_epochs}")
        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val Loss: {val_metrics['loss']:.4f}")
        print(f"  Val F1-Micro: {val_metrics['f1_micro']:.4f}")
        print(f"  Val F1-Macro: {val_metrics['f1_macro']:.4f}")
        print(f"  Val Subset Accuracy: {val_metrics['subset_accuracy']:.4f}")
        print(f"  Val Label Accuracy: {val_metrics['label_accuracy']:.4f}")

        # Early stopping
        if val_metrics['f1_micro'] > best_f1:
            best_f1 = val_metrics['f1_micro']
            patience_counter = 0
            # Sauvegarder le meilleur modèle
            torch.save(model.state_dict(), f'{model_name}_best.pth')
            print(f"  ✅ Nouveau meilleur modèle sauvegardé !")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  ⏹️ Early stopping à l'epoch {epoch+1}")
                break

    # Charger le meilleur modèle
    model.load_state_dict(torch.load(f'{model_name}_best.pth'))

    # Sauvegarder le modèle final en pickle
    with open(f'{model_name}_final.pickle', 'wb') as f:
        pickle.dump({
            'model_state_dict': model.state_dict(),
            'model_class': model.__class__.__name__,
            'history': history,
            'best_f1': best_f1
        }, f)

    print(f"\n✅ Entraînement terminé ! Meilleur F1-Micro: {best_f1:.4f}")
    print(f"📦 Modèle sauvegardé dans '{model_name}_final.pickle'")

    return model, history



⚙️ DÉFINITION DES FONCTIONS D'ENTRAÎNEMENT...


### SECTION 7 : ENTRAÎNEMENT DES MODÈLES

In [11]:
print("\n" + "="*80)
print("🎯 ENTRAÎNEMENT DES MODÈLES")
print("="*80)

# Paramètres d'entraînement
NUM_EPOCHS = 15
VOCAB_SIZE = tokenizer.vocab_size

results = {}


🎯 ENTRAÎNEMENT DES MODÈLES


#### --------------- MODÈLE 1 : LSTM Simple ---------------

In [12]:
print("\n📚 MODÈLE 1 : LSTM Simple")
lstm_model = SimpleLSTM(
    vocab_size=VOCAB_SIZE,
    embedding_dim=300,
    hidden_dim=256,
    num_classes=len(EMOTIONS),
    dropout=0.3
)

lstm_trained, lstm_history = train_model(
    lstm_model,
    train_loader,
    dev_loader,
    num_epochs=NUM_EPOCHS,
    lr=1e-3,
    model_name='lstm_simple'
)

# Évaluation finale sur le test set
test_metrics_lstm = evaluate_model(lstm_trained, test_loader,
                                   nn.BCEWithLogitsLoss(), device)
results['LSTM'] = test_metrics_lstm


📚 MODÈLE 1 : LSTM Simple

🚀 Début de l'entraînement du modèle lstm_simple...
Epoch 1/15
  Train Loss: 0.6888
  Val Loss: 0.6887
  Val F1-Micro: 0.0786
  Val F1-Macro: 0.0299
  Val Subset Accuracy: 0.0000
  Val Label Accuracy: 0.6322
  ✅ Nouveau meilleur modèle sauvegardé !
Epoch 2/15
  Train Loss: 0.6869
  Val Loss: 0.6848
  Val F1-Micro: 0.0786
  Val F1-Macro: 0.0299
  Val Subset Accuracy: 0.0000
  Val Label Accuracy: 0.6322
Epoch 3/15
  Train Loss: 0.6799
  Val Loss: 0.6736
  Val F1-Micro: 0.0811
  Val F1-Macro: 0.0169
  Val Subset Accuracy: 0.0000
  Val Label Accuracy: 0.7973
  ✅ Nouveau meilleur modèle sauvegardé !
Epoch 4/15
  Train Loss: 0.5429
  Val Loss: 0.2995
  Val F1-Micro: 0.0000
  Val F1-Macro: 0.0000
  Val Subset Accuracy: 0.0000
  Val Label Accuracy: 0.9580
Epoch 5/15
  Train Loss: 0.2409
  Val Loss: 0.1962
  Val F1-Micro: 0.0000
  Val F1-Macro: 0.0000
  Val Subset Accuracy: 0.0000
  Val Label Accuracy: 0.9580
Epoch 6/15
  Train Loss: 0.1807
  Val Loss: 0.1636
  Val F1-

#### --------------- MODÈLE 2 : BiLSTM avec Attention ---------------

In [13]:
print("\n📚 MODÈLE 2 : BiLSTM avec Attention")
bilstm_model = BiLSTMAttention(
    vocab_size=VOCAB_SIZE,
    embedding_dim=300,
    hidden_dim=256,
    num_classes=len(EMOTIONS),
    dropout=0.3
)

bilstm_trained, bilstm_history = train_model(
    bilstm_model,
    train_loader,
    dev_loader,
    num_epochs=NUM_EPOCHS,
    lr=1e-3,
    model_name='bilstm_attention'
)

test_metrics_bilstm = evaluate_model(bilstm_trained, test_loader,
                                     nn.BCEWithLogitsLoss(), device)
results['BiLSTM+Attention'] = test_metrics_bilstm


📚 MODÈLE 2 : BiLSTM avec Attention

🚀 Début de l'entraînement du modèle bilstm_attention...
Epoch 1/15
  Train Loss: 0.6920
  Val Loss: 0.6920
  Val F1-Micro: 0.0496
  Val F1-Macro: 0.0253
  Val Subset Accuracy: 0.0000
  Val Label Accuracy: 0.5229
  ✅ Nouveau meilleur modèle sauvegardé !
Epoch 2/15
  Train Loss: 0.6865
  Val Loss: 0.6803
  Val F1-Micro: 0.0668
  Val F1-Macro: 0.0162
  Val Subset Accuracy: 0.0000
  Val Label Accuracy: 0.7669
  ✅ Nouveau meilleur modèle sauvegardé !
Epoch 3/15
  Train Loss: 0.6493
  Val Loss: 0.5662
  Val F1-Micro: 0.0000
  Val F1-Macro: 0.0000
  Val Subset Accuracy: 0.0000
  Val Label Accuracy: 0.9580
Epoch 4/15
  Train Loss: 0.2344
  Val Loss: 0.1603
  Val F1-Micro: 0.0000
  Val F1-Macro: 0.0000
  Val Subset Accuracy: 0.0000
  Val Label Accuracy: 0.9580
Epoch 5/15
  Train Loss: 0.1572
  Val Loss: 0.1518
  Val F1-Micro: 0.0000
  Val F1-Macro: 0.0000
  Val Subset Accuracy: 0.0000
  Val Label Accuracy: 0.9580
  ⏹️ Early stopping à l'epoch 5

✅ Entraîneme

#### --------------- MODÈLE 3 : CNN-BiLSTM Hybride ---------------

In [14]:
print("\n📚 MODÈLE 3 : CNN-BiLSTM Hybride avec Attention")

hybrid_model = CNNBiLSTMAttention(
    vocab_size=VOCAB_SIZE,
    embedding_dim=300,
    hidden_dim=256,
    num_classes=len(EMOTIONS),
    dropout=0.3,
    num_filters=100
).to(device)

hybrid_trained, hybrid_history = train_model(
    hybrid_model,
    train_loader,
    dev_loader,
    num_epochs=NUM_EPOCHS,
    lr=1e-3,
    model_name='cnn_bilstm_hybrid'
)

test_metrics_hybrid = evaluate_model(
    hybrid_trained,
    test_loader,
    nn.BCEWithLogitsLoss(),
    device
)

results['CNN-BiLSTM'] = test_metrics_hybrid


📚 MODÈLE 3 : CNN-BiLSTM Hybride avec Attention

🚀 Début de l'entraînement du modèle cnn_bilstm_hybrid...
Epoch 1/15
  Train Loss: 0.6959
  Val Loss: 0.6959
  Val F1-Micro: 0.0954
  Val F1-Macro: 0.0604
  Val Subset Accuracy: 0.0000
  Val Label Accuracy: 0.3542
  ✅ Nouveau meilleur modèle sauvegardé !
Epoch 2/15
  Train Loss: 0.6882
  Val Loss: 0.6785
  Val F1-Micro: 0.0902
  Val F1-Macro: 0.0192
  Val Subset Accuracy: 0.0000
  Val Label Accuracy: 0.8006
Epoch 3/15
  Train Loss: 0.5037
  Val Loss: 0.2505
  Val F1-Micro: 0.0000
  Val F1-Macro: 0.0000
  Val Subset Accuracy: 0.0000
  Val Label Accuracy: 0.9580
Epoch 4/15
  Train Loss: 0.1925
  Val Loss: 0.1617
  Val F1-Micro: 0.0000
  Val F1-Macro: 0.0000
  Val Subset Accuracy: 0.0000
  Val Label Accuracy: 0.9580
  ⏹️ Early stopping à l'epoch 4

✅ Entraînement terminé ! Meilleur F1-Micro: 0.0954
📦 Modèle sauvegardé dans 'cnn_bilstm_hybrid_final.pickle'


#### --------------- MODÈLE 4 : BERT ---------------

In [15]:
print("\n📚 MODÈLE 4 : BERT Fine-tuned")
bert_model = BERTEmotionClassifier(
    num_classes=len(EMOTIONS),
    dropout=0.3
)

bert_trained, bert_history = train_model(
    bert_model,
    train_loader,
    dev_loader,
    num_epochs=NUM_EPOCHS,
    lr=2e-5,
    model_name='bert_finetuned'
)

test_metrics_bert = evaluate_model(bert_trained, test_loader,
                                   nn.BCEWithLogitsLoss(), device)
results['BERT'] = test_metrics_bert


📚 MODÈLE 4 : BERT Fine-tuned


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]


🚀 Début de l'entraînement du modèle bert_finetuned...
Epoch 1/15
  Train Loss: 0.6926
  Val Loss: 0.6883
  Val F1-Micro: 0.0603
  Val F1-Macro: 0.0441
  Val Subset Accuracy: 0.0000
  Val Label Accuracy: 0.5042
  ✅ Nouveau meilleur modèle sauvegardé !
Epoch 2/15
  Train Loss: 0.6877
  Val Loss: 0.6785
  Val F1-Micro: 0.0687
  Val F1-Macro: 0.0472
  Val Subset Accuracy: 0.0000
  Val Label Accuracy: 0.5562
  ✅ Nouveau meilleur modèle sauvegardé !
Epoch 3/15
  Train Loss: 0.6755
  Val Loss: 0.6624
  Val F1-Micro: 0.0713
  Val F1-Macro: 0.0459
  Val Subset Accuracy: 0.0000
  Val Label Accuracy: 0.6221
  ✅ Nouveau meilleur modèle sauvegardé !
Epoch 4/15
  Train Loss: 0.6563
  Val Loss: 0.6371
  Val F1-Micro: 0.0607
  Val F1-Macro: 0.0378
  Val Subset Accuracy: 0.0000
  Val Label Accuracy: 0.6937
Epoch 5/15
  Train Loss: 0.6246
  Val Loss: 0.5953
  Val F1-Micro: 0.0523
  Val F1-Macro: 0.0254
  Val Subset Accuracy: 0.0000
  Val Label Accuracy: 0.7644
Epoch 6/15
  Train Loss: 0.5768
  Val Loss

### SECTION 8 : COMPARAISON DES MODÈLES

In [16]:
print("\n" + "="*80)
print("📊 COMPARAISON DES PERFORMANCES")
print("="*80)

# Créer un DataFrame de comparaison
comparison_df = pd.DataFrame(results).T
comparison_df = comparison_df.round(4)

print("\n📈 Tableau récapitulatif des métriques :")
print(comparison_df.to_string())

# Visualisation comparative
metrics_to_plot = ['f1_micro', 'f1_macro', 'precision_micro', 'recall_micro']
fig = go.Figure()

for metric in metrics_to_plot:
    fig.add_trace(go.Bar(
        name=metric.replace('_', ' ').title(),
        x=list(results.keys()),
        y=[results[model][metric] for model in results.keys()],
    ))

fig.update_layout(
    title='Comparaison des performances des modèles',
    xaxis_title='Modèle',
    yaxis_title='Score',
    barmode='group',
    height=500
)
fig.show()

# Trouver le meilleur modèle
best_model_name = max(results, key=lambda x: results[x]['f1_micro'])
print(f"\n🏆 Meilleur modèle : {best_model_name}")
print(f"F1-Micro : {results[best_model_name]['f1_micro']:.4f}")



📊 COMPARAISON DES PERFORMANCES

📈 Tableau récapitulatif des métriques :
                    loss  precision_micro  recall_micro  f1_micro  precision_macro  recall_macro  f1_macro  hamming_loss  subset_accuracy  label_accuracy  auc_roc
LSTM              0.6736           0.0468        0.2005    0.0758           0.0084        0.1786    0.0159        0.2035              0.0          0.7965   0.5000
BiLSTM+Attention  0.6803           0.0400        0.1994    0.0666           0.0085        0.2078    0.0161        0.2329              0.0          0.7671   0.5256
CNN-BiLSTM        0.6959           0.0508        0.8199    0.0957           0.0342        0.6696    0.0605        0.6456              0.0          0.3544   0.5002
BERT              0.6623           0.0396        0.3468    0.0710           0.0336        0.3743    0.0465        0.3778              0.0          0.6222   0.5132



🏆 Meilleur modèle : CNN-BiLSTM
F1-Micro : 0.0957


### SECTION 9 : ÉTUDE D'ABLATION (CNN-BiLSTM)

In [17]:
print("\n" + "="*80)
print("🔬 ÉTUDE D'ABLATION - CNN-BiLSTM")
print("="*80)

ablation_results = {}

# 1. Sans attention
class CNNBiLSTMNoAttention(nn.Module):
    def __init__(self, vocab_size, embedding_dim=300, hidden_dim=256,
                 num_classes=28, dropout=0.3, num_filters=100):
        super(CNNBiLSTMNoAttention, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        # Utiliser le même padding pour toutes les convolutions
        self.conv1 = nn.Conv1d(embedding_dim, num_filters, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(embedding_dim, num_filters, kernel_size=4, padding=2)
        self.conv3 = nn.Conv1d(embedding_dim, num_filters, kernel_size=5, padding=2)

        self.bilstm = nn.LSTM(
            num_filters * 3,
            hidden_dim,
            batch_first=True,
            bidirectional=True,
            dropout=dropout,
            num_layers=2
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids, attention_mask=None):
        embedded = self.embedding(input_ids)
        embedded_t = embedded.transpose(1, 2)

        conv1_out = F.relu(self.conv1(embedded_t))
        conv2_out = F.relu(self.conv2(embedded_t))
        conv3_out = F.relu(self.conv3(embedded_t))

        # Trouver la taille minimale et découper toutes les sorties
        min_len = min(conv1_out.size(2), conv2_out.size(2), conv3_out.size(2))
        conv1_out = conv1_out[:, :, :min_len]
        conv2_out = conv2_out[:, :, :min_len]
        conv3_out = conv3_out[:, :, :min_len]

        conv_out = torch.cat([conv1_out, conv2_out, conv3_out], dim=1)
        conv_out = conv_out.transpose(1, 2)

        lstm_out, (hidden, _) = self.bilstm(conv_out)

        # Utiliser le dernier état caché au lieu de l'attention
        output = torch.cat([hidden[-2], hidden[-1]], dim=1)
        output = self.dropout(output)
        logits = self.fc(output)

        return logits

print("\n🔬 Test 1 : Sans mécanisme d'attention")
model_no_attn = CNNBiLSTMNoAttention(
    vocab_size=VOCAB_SIZE,
    embedding_dim=300,
    hidden_dim=256,
    num_classes=len(EMOTIONS),
    dropout=0.3
)

_, _ = train_model(
    model_no_attn,
    train_loader,
    dev_loader,
    num_epochs=3,
    lr=1e-3,
    model_name='ablation_no_attention'
)

metrics_no_attn = evaluate_model(model_no_attn, test_loader,
                                 nn.BCEWithLogitsLoss(), device)
ablation_results['Sans Attention'] = metrics_no_attn

# 2. Sans CNN (BiLSTM seulement)
print("\n🔬 Test 2 : Sans CNN (BiLSTM seul)")
metrics_bilstm_only = test_metrics_bilstm  # Déjà calculé
ablation_results['Sans CNN'] = metrics_bilstm_only

# 3. Sans BiLSTM (CNN seulement)
class CNNOnly(nn.Module):
    def __init__(self, vocab_size, embedding_dim=300, num_classes=28,
                 dropout=0.3, num_filters=100):
        super(CNNOnly, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.conv1 = nn.Conv1d(embedding_dim, num_filters, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(embedding_dim, num_filters, kernel_size=4, padding=2)
        self.conv3 = nn.Conv1d(embedding_dim, num_filters, kernel_size=5, padding=2)

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * 3, num_classes)

    def forward(self, input_ids, attention_mask=None):
        embedded = self.embedding(input_ids)
        embedded_t = embedded.transpose(1, 2)

        conv1_out = F.relu(self.conv1(embedded_t))
        conv2_out = F.relu(self.conv2(embedded_t))
        conv3_out = F.relu(self.conv3(embedded_t))

        # Max pooling
        conv1_pool = F.max_pool1d(conv1_out, conv1_out.size(2)).squeeze(2)
        conv2_pool = F.max_pool1d(conv2_out, conv2_out.size(2)).squeeze(2)
        conv3_pool = F.max_pool1d(conv3_out, conv3_out.size(2)).squeeze(2)

        pooled = torch.cat([conv1_pool, conv2_pool, conv3_pool], dim=1)
        output = self.dropout(pooled)
        logits = self.fc(output)

        return logits

print("\n🔬 Test 3 : Sans BiLSTM (CNN seul)")
model_cnn_only = CNNOnly(
    vocab_size=VOCAB_SIZE,
    embedding_dim=300,
    num_classes=len(EMOTIONS),
    dropout=0.3
)

_, _ = train_model(
    model_cnn_only,
    train_loader,
    dev_loader,
    num_epochs=3,
    lr=1e-3,
    model_name='ablation_cnn_only'
)

metrics_cnn_only = evaluate_model(model_cnn_only, test_loader,
                                  nn.BCEWithLogitsLoss(), device)
ablation_results['Sans BiLSTM'] = metrics_cnn_only

# Modèle complet (référence)
ablation_results['Modèle Complet'] = test_metrics_hybrid

# Afficher les résultats de l'ablation
print("\n📊 RÉSULTATS DE L'ÉTUDE D'ABLATION :")
ablation_df = pd.DataFrame(ablation_results).T
ablation_df = ablation_df.round(4)
print(ablation_df[['f1_micro', 'f1_macro', 'precision_micro', 'recall_micro']].to_string())

# Visualisation
fig = go.Figure([go.Bar(
    x=list(ablation_results.keys()),
    y=[ablation_results[k]['f1_micro'] for k in ablation_results.keys()],
    marker_color=['red', 'orange', 'orange', 'green']
)])
fig.update_layout(
    title='Impact des composants sur la performance (F1-Micro)',
    xaxis_title='Configuration',
    yaxis_title='F1-Micro Score',
    height=400
)
fig.show()


🔬 ÉTUDE D'ABLATION - CNN-BiLSTM

🔬 Test 1 : Sans mécanisme d'attention

🚀 Début de l'entraînement du modèle ablation_no_attention...
Epoch 1/3
  Train Loss: 0.6937
  Val Loss: 0.6937
  Val F1-Micro: 0.1015
  Val F1-Macro: 0.0508
  Val Subset Accuracy: 0.0000
  Val Label Accuracy: 0.5376
  ✅ Nouveau meilleur modèle sauvegardé !
Epoch 2/3
  Train Loss: 0.3720
  Val Loss: 0.1630
  Val F1-Micro: 0.0000
  Val F1-Macro: 0.0000
  Val Subset Accuracy: 0.0000
  Val Label Accuracy: 0.9580
Epoch 3/3
  Train Loss: 0.1548
  Val Loss: 0.1499
  Val F1-Micro: 0.0000
  Val F1-Macro: 0.0000
  Val Subset Accuracy: 0.0000
  Val Label Accuracy: 0.9580

✅ Entraînement terminé ! Meilleur F1-Micro: 0.1015
📦 Modèle sauvegardé dans 'ablation_no_attention_final.pickle'

🔬 Test 2 : Sans CNN (BiLSTM seul)

🔬 Test 3 : Sans BiLSTM (CNN seul)

🚀 Début de l'entraînement du modèle ablation_cnn_only...
Epoch 1/3
  Train Loss: 0.8786
  Val Loss: 0.8613
  Val F1-Micro: 0.0764
  Val F1-Macro: 0.0537
  Val Subset Accuracy:

### SECTION 10 : ANALYSE D'EXPLICABILITÉ

#### 1- PRÉPARATION DU MODÈLE EXPLICABLE

In [18]:
print("\n" + "="*80)
print("🔧 PRÉPARATION DU MODÈLE POUR L'EXPLICABILITÉ")
print("="*80)

class ExplainableEmotionModel:
    """
    Wrapper qui rend modèle compatible avec LIME et SHAP
    """

    def __init__(self, model, tokenizer, device, emotions):

        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.emotions = emotions
        self.model.eval()

        print("✅ Modèle explicable initialisé")
        print(f"   Device: {device}")
        print(f"   Nombre d'émotions: {len(emotions)}")

    def predict_proba(self, texts):

        # Gérer le cas d'un seul texte
        if isinstance(texts, str):
            texts = [texts]

        all_probs = []

        # Prédire pour chaque texte
        for text in texts:
            encoding = self.tokenizer.encode_plus(
                text,
                add_special_tokens=True,
                max_length=128,
                padding='max_length',
                truncation=True,
                return_attention_mask=True,
                return_tensors='pt'
            )

            input_ids = encoding['input_ids'].to(self.device)
            attention_mask = encoding['attention_mask'].to(self.device)

            with torch.no_grad():
                outputs = self.model(input_ids, attention_mask)
                probs = torch.sigmoid(outputs).cpu().numpy()[0]

            all_probs.append(probs)

        return np.array(all_probs)

    def predict_single(self, text):
        probs = self.predict_proba(text)[0]

        # Retourner les top 5 émotions
        top_indices = np.argsort(probs)[-5:][::-1]
        results = {
            self.emotions[idx]: float(probs[idx])
            for idx in top_indices
        }
        return results

# Créer le modèle explicable
explainable_model = ExplainableEmotionModel(
    model=bert_trained,
    tokenizer=tokenizer,
    device=device,
    emotions=EMOTIONS
)

# Test rapide
print("\n🧪 Test du modèle explicable :")
test_text = "I am so happy today!"
predictions = explainable_model.predict_single(test_text)
print(f"Texte : '{test_text}'")
print("Prédictions :")
for emotion, prob in predictions.items():
    print(f"  {emotion}: {prob:.3f}")



🔧 PRÉPARATION DU MODÈLE POUR L'EXPLICABILITÉ
✅ Modèle explicable initialisé
   Device: cuda
   Nombre d'émotions: 28

🧪 Test du modèle explicable :
Texte : 'I am so happy today!'
Prédictions :
  caring: 0.654
  disgust: 0.578
  confusion: 0.561
  fear: 0.560
  curiosity: 0.559


#### 2- LIME - EXPLICATIONS PAR PERTURBATION

In [19]:
import lime
from lime.lime_text import LimeTextExplainer

print("\n" + "="*80)
print("🍋 ANALYSE AVEC LIME")
print("="*80)

# Initialiser l'explicateur LIME
lime_explainer = LimeTextExplainer(
    class_names=EMOTIONS,
    bow=False,
    random_state=42
)

print("✅ Explicateur LIME initialisé")

def explain_with_lime(text, num_features=10, num_samples=1000, show_top_emotions=3):

    print("\n" + "┌" + "─"*78 + "┐")
    print(f"│ 📝 ANALYSE LIME : {text[:60]:<60} │")
    print("└" + "─"*78 + "┘")

    print(f"\n⏳ Génération de {num_samples} variations du texte...")
    print("   (cela peut prendre 10-30 secondes)")

    # Générer l'explication
    explanation = lime_explainer.explain_instance(
        text,
        explainable_model.predict_proba,
        num_features=num_features,
        num_samples=num_samples,
        top_labels=show_top_emotions
    )

    # Obtenir les prédictions
    probs = explainable_model.predict_proba(text)[0]
    top_indices = np.argsort(probs)[-show_top_emotions:][::-1]

    print(f"\n✅ Analyse terminée !")
    print(f"\n{'='*80}")
    print("🎯 PRÉDICTIONS DU MODÈLE")
    print(f"{'='*80}")

    for rank, idx in enumerate(top_indices, 1):
        emotion = EMOTIONS[idx]
        confidence = probs[idx]
        bar = "█" * int(confidence * 50)
        print(f"{rank}. {emotion:<15} {confidence:.3f}  {bar}")

    # Expliquer chaque émotion
    for idx in top_indices:
        emotion = EMOTIONS[idx]
        confidence = probs[idx]

        print(f"\n{'='*80}")
        print(f"📊 EXPLICATION POUR : {emotion.upper()} (confiance: {confidence:.3f})")
        print(f"{'='*80}")

        # Récupérer les poids des mots
        word_weights = explanation.as_list(label=idx)

        print(f"\n┌{'─'*78}┐")
        print(f"│ {'Mot':<25} {'Impact':<15} {'Contribution':<35} │")
        print(f"├{'─'*78}┤")

        for word, weight in word_weights[:num_features]:
            # Déterminer l'impact
            if weight > 0:
                impact = "🟢 POSITIF"
                color_bar = "█"
            else:
                impact = "🔴 NÉGATIF"
                color_bar = "▓"

            # Créer une barre visuelle
            bar_length = int(abs(weight) * 30)
            bar = color_bar * bar_length

            print(f"│ {word:<25} {impact:<15} {bar:<35} │")
            print(f"│ {'':25} {'':15} {weight:>+.4f}{'':28} │")
            print(f"├{'─'*78}┤")

        print(f"└{'─'*78}┘")

        # Interprétation
        positive_words = [w for w, wt in word_weights if wt > 0]
        negative_words = [w for w, wt in word_weights if wt < 0]

        print(f"\n💡 INTERPRÉTATION :")
        if positive_words:
            print(f"   ✓ Mots qui FAVORISENT '{emotion}' : {', '.join(positive_words[:5])}")
        if negative_words:
            print(f"   ✗ Mots qui DÉFAVORISENT '{emotion}' : {', '.join(negative_words[:5])}")

    return explanation

# Exemples d'analyse LIME
print("\n" + "🔬 DÉMONSTRATION LIME SUR DES EXEMPLES RÉELS")
print("="*80)

examples_lime = [
    "I am so happy and excited about this amazing opportunity!",
    "This is terrible and makes me feel angry and disappointed.",
    "I'm really worried about what might happen next.",
]

for i, text in enumerate(examples_lime, 1):
    print(f"\n\n{'#'*80}")
    print(f"# EXEMPLE {i}/3")
    print(f"{'#'*80}")
    explain_with_lime(text, num_features=8, num_samples=500)



🍋 ANALYSE AVEC LIME
✅ Explicateur LIME initialisé

🔬 DÉMONSTRATION LIME SUR DES EXEMPLES RÉELS


################################################################################
# EXEMPLE 1/3
################################################################################

┌──────────────────────────────────────────────────────────────────────────────┐
│ 📝 ANALYSE LIME : I am so happy and excited about this amazing opportunity!    │
└──────────────────────────────────────────────────────────────────────────────┘

⏳ Génération de 500 variations du texte...
   (cela peut prendre 10-30 secondes)

✅ Analyse terminée !

🎯 PRÉDICTIONS DU MODÈLE
1. caring          0.645  ████████████████████████████████
2. fear            0.587  █████████████████████████████
3. disgust         0.576  ████████████████████████████

📊 EXPLICATION POUR : CARING (confiance: 0.645)

┌──────────────────────────────────────────────────────────────────────────────┐
│ Mot                       Impact          Contribu

#### 3- ISUALISATION D'ATTENTION (HEATMAP)

In [20]:
print("\n" + "="*80)
print(" VISUALISATION DES POIDS D'ATTENTION")
print("="*80)

def get_attention_weights(model, text, tokenizer, device):
    model.eval()  # We should be in eval mode for inference

    # Tokenization
    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # Create a detached clone with requires_grad
    with torch.no_grad():
        embeddings = model.bert.embeddings(input_ids)

    # Clone and set requires_grad on the clone
    embeddings = embeddings.clone().detach().requires_grad_(True)

    # Forward pass with embeddings
    outputs = model.bert(
        inputs_embeds=embeddings,
        attention_mask=attention_mask
    )

    pooled_output = outputs.pooler_output
    logits = model.classifier(model.dropout(pooled_output))
    probs = torch.sigmoid(logits)

    # Calculate gradients for the top predicted emotion
    top_emotion_idx = torch.argmax(probs[0])

    # Zero gradients before backward pass
    if embeddings.grad is not None:
        embeddings.grad.zero_()

    # Backward pass
    probs[0, top_emotion_idx].backward(retain_graph=True)

    # Get gradients (using absolute value or norm)
    if embeddings.grad is not None:
        gradients = embeddings.grad[0].abs().sum(dim=-1).cpu().numpy()
    else:
        # Fallback: use attention weights if gradients are not available
        with torch.no_grad():
            outputs = model.bert(input_ids, attention_mask=attention_mask)
            # Get attention from last layer (simplified approach)
            gradients = outputs.attentions[-1].mean(dim=1)[0].mean(dim=0).cpu().numpy()

    # Get tokens
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].cpu())

    # Clean (remove padding, CLS, SEP)
    clean_tokens = []
    clean_weights = []

    for token, weight in zip(tokens, gradients):
        if token not in ['[PAD]', '[CLS]', '[SEP]']:
            clean_tokens.append(token.replace('##', ''))
            clean_weights.append(float(weight))

    # Normalize weights between 0 and 1
    if len(clean_weights) > 0:
        if max(clean_weights) > 0:
            max_weight = max(clean_weights)
            clean_weights = [w / max_weight for w in clean_weights]
        else:
            # If all weights are zero, set to small value
            clean_weights = [0.1] * len(clean_weights)

    return clean_tokens, clean_weights, probs[0].detach().cpu().numpy()

def visualize_attention(text, model, tokenizer, device):
    print(f"\n Texte analysé : '{text}'")

    tokens, weights, probs = get_attention_weights(model, text, tokenizer, device)

    # Top 3 emotions
    top_indices = np.argsort(probs)[-3:][::-1]

    print(f"\n Émotions prédites :")
    for idx in top_indices:
        print(f"   {EMOTIONS[idx]}: {probs[idx]:.3f}")

    # Create heatmap
    print(f"\n🔥 HEATMAP D'ATTENTION (importance de chaque mot) :\n")

    print("┌" + "─"*78 + "┐")
    print("│ Plus la couleur est intense, plus le mot est important pour la prédiction │")
    print("└" + "─"*78 + "┘\n")

    for token, weight in zip(tokens, weights):
        # Create visual bar
        bar_length = int(weight * 40)
        bar = "█" * bar_length

        # Color based on importance
        if weight > 0.7:
            symbol = "🔴"
        elif weight > 0.4:
            symbol = "🟡"
        else:
            symbol = "🟢"

        print(f"{symbol} {token:<15} {bar:<40} {weight:.3f}")

    # Plotly visualization
    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=tokens,
        y=weights,
        marker=dict(
            color=weights,
            colorscale='Reds',
            showscale=True,
            colorbar=dict(title="Importance")
        ),
        text=[f"{w:.2f}" for w in weights],
        textposition='outside'
    ))

    fig.update_layout(
        title=f"Poids d'Attention pour : '{text[:50]}...'",
        xaxis_title="Mots",
        yaxis_title="Poids d'attention (normalisé)",
        height=500,
        xaxis_tickangle=-45
    )

    fig.show()

    return tokens, weights

# Alternative simpler approach without gradients (using attention weights directly)
def get_attention_simple(model, text, tokenizer, device):
    model.eval()

    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model.bert(input_ids, attention_mask=attention_mask, output_attentions=True)

        # Get attention weights from all layers
        attentions = outputs.attentions  # Tuple of attention matrices

        # Average attention across all layers and heads
        # attentions[-1] is the last layer: shape (batch, heads, seq_len, seq_len)
        last_layer_attention = attentions[-1]

        # Average across attention heads
        avg_attention = last_layer_attention.mean(dim=1)[0]  # shape: (seq_len, seq_len)

        # We can look at attention from CLS token to other tokens
        cls_attention = avg_attention[0]  # attention from CLS token to all tokens

        # Or average attention received by each token
        token_importance = avg_attention.mean(dim=0).cpu().numpy()

    # Get tokens
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].cpu())

    # Clean tokens
    clean_tokens = []
    clean_weights = []

    for i, (token, weight) in enumerate(zip(tokens, token_importance)):
        if token not in ['[PAD]', '[CLS]', '[SEP]']:
            clean_tokens.append(token.replace('##', ''))
            clean_weights.append(float(weight))

    # Get predictions
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        probs = torch.sigmoid(outputs).cpu().numpy()[0]

    return clean_tokens, clean_weights, probs

# Exemples de visualisation d'attention
print("\n🔬 DÉMONSTRATION DE LA VISUALISATION D'ATTENTION")
print("="*80)

attention_examples = [
    "I love this so much it makes me incredibly happy!",
    "This is disgusting and infuriating, I can't believe it.",
    "I'm confused and curious about what this might mean.",
]

for i, text in enumerate(attention_examples, 1):
    print(f"\n\n{'='*80}")
    print(f"EXEMPLE {i}/3")
    print(f"{'='*80}")
    visualize_attention(text, bert_trained, tokenizer, device)


 VISUALISATION DES POIDS D'ATTENTION

🔬 DÉMONSTRATION DE LA VISUALISATION D'ATTENTION


EXEMPLE 1/3

 Texte analysé : 'I love this so much it makes me incredibly happy!'

 Émotions prédites :
   caring: 0.662
   disgust: 0.586
   fear: 0.569

🔥 HEATMAP D'ATTENTION (importance de chaque mot) :

┌──────────────────────────────────────────────────────────────────────────────┐
│ Plus la couleur est intense, plus le mot est important pour la prédiction │
└──────────────────────────────────────────────────────────────────────────────┘

🟢 i               ███████████                              0.285
🟢 love            ███████████████                          0.386
🟡 this            ██████████████████████                   0.565
🟢 so              ██████████████                           0.360
🟢 much            █████████████                            0.332
🟢 it              █████████                                0.232
🟢 makes           ████████████                             0.318
🟡 me    



EXEMPLE 2/3

 Texte analysé : 'This is disgusting and infuriating, I can't believe it.'

 Émotions prédites :
   caring: 0.638
   confusion: 0.571
   disgust: 0.568

🔥 HEATMAP D'ATTENTION (importance de chaque mot) :

┌──────────────────────────────────────────────────────────────────────────────┐
│ Plus la couleur est intense, plus le mot est important pour la prédiction │
└──────────────────────────────────────────────────────────────────────────────┘

🟡 this            █████████████████████                    0.528
🟢 is              ██████████████                           0.369
🔴 disgusting      ████████████████████████████████████████ 1.000
🟡 and             ██████████████████                       0.465
🟡 in              █████████████████████                    0.540
🟡 fur             ███████████████████████████              0.686
🟡 iating          ████████████████████████                 0.614
🟡 ,               ██████████████████████████               0.664
🟢 i               █



EXEMPLE 3/3

 Texte analysé : 'I'm confused and curious about what this might mean.'

 Émotions prédites :
   caring: 0.616
   curiosity: 0.570
   fear: 0.554

🔥 HEATMAP D'ATTENTION (importance de chaque mot) :

┌──────────────────────────────────────────────────────────────────────────────┐
│ Plus la couleur est intense, plus le mot est important pour la prédiction │
└──────────────────────────────────────────────────────────────────────────────┘

🟡 i               ████████████████                         0.408
🔴 '               ██████████████████████████████           0.770
🟡 m               █████████████████                        0.447
🔴 confused        ████████████████████████████████████████ 1.000
🔴 and             █████████████████████████████            0.744
🔴 curious         ██████████████████████████████           0.755
🟡 about           █████████████████                        0.449
🟡 what            ████████████████                         0.410
🟡 this            ███████

#### 4- ANALYSE APPROFONDIE DES ERREURS




In [21]:
from plotly.subplots import make_subplots

print("\n" + "="*80)
print("❌ ANALYSE DES ERREURS DE CLASSIFICATION")
print("="*80)

# Collecter toutes les prédictions sur le test set
print("\n⏳ Collecte des prédictions sur le test set...")

bert_trained.eval()
all_predictions = []
all_labels = []
all_probs = []
all_texts = []

with torch.no_grad():
    for i, batch in enumerate(test_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']

        outputs = bert_trained(input_ids, attention_mask)
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()

        all_predictions.append(preds.cpu().numpy())
        all_labels.append(labels.numpy())
        all_probs.append(probs.cpu().numpy())

all_predictions = np.vstack(all_predictions)
all_labels = np.vstack(all_labels)
all_probs = np.vstack(all_probs)

print("✅ Prédictions collectées")

# Calculer les métriques par émotion
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    all_labels,
    all_predictions,
    average=None,
    zero_division=0
)

# Créer un DataFrame des performances
emotion_perf = pd.DataFrame({
    'Emotion': EMOTIONS,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support.astype(int)
})

emotion_perf = emotion_perf.sort_values('F1-Score', ascending=True)

print("\n📊 PERFORMANCES PAR ÉMOTION (10 PLUS DIFFICILES) :")
print("="*80)
print(emotion_perf.head(10).to_string(index=False))

print("\n📊 PERFORMANCES PAR ÉMOTION (10 PLUS FACILES) :")
print("="*80)
print(emotion_perf.tail(10).to_string(index=False))

# Visualisation
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("F1-Score par Émotion", "Support par Émotion"),
    vertical_spacing=0.15
)

# F1-Score
sorted_emotions = emotion_perf.sort_values('F1-Score', ascending=False)
colors = ['red' if f1 < 0.3 else 'orange' if f1 < 0.5 else 'green'
          for f1 in sorted_emotions['F1-Score']]

fig.add_trace(
    go.Bar(
        x=sorted_emotions['Emotion'],
        y=sorted_emotions['F1-Score'],
        marker_color=colors,
        text=sorted_emotions['F1-Score'].round(3),
        textposition='outside',
        name='F1-Score'
    ),
    row=1, col=1
)

# Support
fig.add_trace(
    go.Bar(
        x=sorted_emotions['Emotion'],
        y=sorted_emotions['Support'],
        marker_color='lightblue',
        name='Support'
    ),
    row=2, col=1
)

fig.update_layout(
    title_text="Analyse Détaillée des Performances",
    height=900,
    showlegend=False
)

fig.update_xaxes(tickangle=-45, row=1, col=1)
fig.update_xaxes(tickangle=-45, row=2, col=1)

fig.show()

# Matrice de confusion pour les émotions les plus fréquentes
print("\n🔍 ANALYSE DES CONFUSIONS ENTRE ÉMOTIONS")
print("="*80)

# Identifier les erreurs communes
error_matrix = np.zeros((len(EMOTIONS), len(EMOTIONS)))

for true_labels, pred_labels in zip(all_labels, all_predictions):
    true_indices = np.where(true_labels == 1)[0]
    pred_indices = np.where(pred_labels == 1)[0]

    # Faux positifs
    for pred_idx in pred_indices:
        if pred_idx not in true_indices:
            for true_idx in true_indices:
                error_matrix[true_idx, pred_idx] += 1

# Top 10 confusions
top_confusions = []
for i in range(len(EMOTIONS)):
    for j in range(len(EMOTIONS)):
        if i != j and error_matrix[i, j] > 0:
            top_confusions.append((
                EMOTIONS[i],
                EMOTIONS[j],
                int(error_matrix[i, j])
            ))

top_confusions = sorted(top_confusions, key=lambda x: x[2], reverse=True)[:10]

print("\n🔄 TOP 10 DES CONFUSIONS :")
print(f"{'Vraie Émotion':<20} {'Prédite à tort':<20} {'Occurrences'}")
print("-" * 60)
for true_em, pred_em, count in top_confusions:
    print(f"{true_em:<20} → {pred_em:<20} {count:>5} fois")

# Trouver des exemples d'erreurs
print("\n\n📝 EXEMPLES CONCRETS D'ERREURS AVEC EXPLICATIONS")
print("="*80)

# Trouver des erreurs intéressantes
errors_found = 0
max_errors_to_show = 3

for idx in range(len(all_labels)):
    if errors_found >= max_errors_to_show:
        break

    true_labels = all_labels[idx]
    pred_labels = all_predictions[idx]

    # Vérifier si c'est une erreur
    if not np.array_equal(true_labels, pred_labels):
        # Obtenir le texte correspondant
        text_idx = idx
        if text_idx < len(df_test):
            text = df_test.iloc[text_idx]['text']

            true_emotions = [EMOTIONS[i] for i in range(len(EMOTIONS)) if true_labels[i] == 1]
            pred_emotions = [EMOTIONS[i] for i in range(len(EMOTIONS)) if pred_labels[i] == 1]

            errors_found += 1

            print(f"\n{'─'*80}")
            print(f"ERREUR #{errors_found}")
            print(f"{'─'*80}")
            print(f"📝 Texte : {text}")
            print(f"\n✅ Émotions réelles : {', '.join(true_emotions)}")
            print(f"❌ Émotions prédites : {', '.join(pred_emotions)}")

            # Analyser avec LIME pourquoi le modèle s'est trompé
            print(f"\n🔍 Analyse LIME de cette erreur :")
            explain_with_lime(text, num_features=6, num_samples=300, show_top_emotions=2)



❌ ANALYSE DES ERREURS DE CLASSIFICATION

⏳ Collecte des prédictions sur le test set...
✅ Prédictions collectées

📊 PERFORMANCES PAR ÉMOTION (10 PLUS DIFFICILES) :
      Emotion  Precision   Recall  F1-Score  Support
    amusement   0.000000 0.000000  0.000000      264
     approval   0.000000 0.000000  0.000000      351
embarrassment   0.000000 0.000000  0.000000       37
    gratitude   0.000000 0.000000  0.000000      352
     surprise   0.000000 0.000000  0.000000      141
       relief   0.000000 0.000000  0.000000       11
        pride   0.000000 0.000000  0.000000       16
         love   0.000000 0.000000  0.000000      238
        grief   0.001926 0.500000  0.003836        6
  nervousness   0.004574 0.347826  0.009029       23

📊 PERFORMANCES PAR ÉMOTION (10 PLUS FACILES) :
    Emotion  Precision   Recall  F1-Score  Support
  confusion   0.027749 0.869281  0.053781      153
 excitement   0.035545 0.145631  0.057143      103
    sadness   0.031298 0.525641  0.059078      156
 


🔍 ANALYSE DES CONFUSIONS ENTRE ÉMOTIONS

🔄 TOP 10 DES CONFUSIONS :
Vraie Émotion        Prédite à tort       Occurrences
------------------------------------------------------------
neutral              → caring                1783 fois
neutral              → fear                  1783 fois
neutral              → disappointment        1771 fois
neutral              → curiosity             1759 fois
neutral              → disgust               1735 fois
neutral              → confusion             1555 fois
neutral              → anger                 1290 fois
neutral              → remorse                886 fois
neutral              → sadness                875 fois
neutral              → joy                    789 fois


📝 EXEMPLES CONCRETS D'ERREURS AVEC EXPLICATIONS

────────────────────────────────────────────────────────────────────────────────
ERREUR #1
────────────────────────────────────────────────────────────────────────────────
📝 Texte : I’m really sorry about your situat